In [1]:
import json, glob, re, pycm, pandas as pd, numpy as np, scipy.stats as stats
from sklearn.metrics import f1_score
from typing import Tuple, List
from subsampling import subsample_statistic_standard_error
import warnings

In [2]:
RUN_VERSION = "v31"

In [4]:
data = []
for file in glob.glob(f"experiments/{RUN_VERSION}/*.json"):
    match = re.match(r'experiments/.+/(.+)-(baseline|zero|few)-(gpqa|simpleqa|mmlu-pro).json', file)
    model = match.group(1)
    prompt = match.group(2)
    dataset = match.group(3)
    df_run = pd.DataFrame(json.load(open(file, 'r')))
    keys = {
        "Judge Model": model,
        "Prompt": prompt,
        "Dataset": dataset
    }
    n = len(df_run)
    wk_v_counts = df_run["wk_v"].value_counts()
    if not 'e' in wk_v_counts:
        wk_v_counts['e'] = 0.0
    wk_v_freq = (wk_v_counts / n).to_dict()
    addl_stats = { 
        'Time (mean)': df_run['execution_time'].mean(),
        'Time (stdev)': df_run['execution_time'].std(),
        'Tokens (mean)': df_run['tokens_used'].mean(),
        'Tokens (stdev)': df_run['tokens_used'].std(),
        'Coverage': (n - wk_v_counts['e']) / n
    }
    data.append({ **keys, **wk_v_freq, **addl_stats })
df1 = pd.DataFrame.from_records(data)
df1 = df1.round(3)
df1 = df1.sort_values(["Judge Model", "Dataset", "Prompt"])
df1

,Judge Model,Prompt,Dataset,f,t,e,Time (mean),Time (stdev),Tokens (mean),Tokens (stdev),Coverage
33,claude-3-5-haiku-20241022,baseline,gpqa,0.418,0.582,0.000,14.249,2.235,1346.538,377.096,1.000
14,claude-3-5-haiku-20241022,few,gpqa,0.408,0.590,0.002,19.476,1.553,4829.188,327.028,0.998
19,claude-3-5-haiku-20241022,zero,gpqa,0.338,0.662,0.000,19.696,2.435,2070.332,328.058,1.000
34,claude-3-5-haiku-20241022,baseline,simpleqa,0.618,0.382,0.000,6.687,1.598,489.725,91.756,1.000
26,claude-3-5-haiku-20241022,few,simpleqa,0.840,0.160,0.000,17.023,1.548,4332.010,63.458,1.000
31,claude-3-5-haiku-20241022,zero,simpleqa,0.762,0.238,0.000,16.287,3.868,1505.905,93.596,1.000
24,claude-3-5-sonnet-20241022,baseline,gpqa,0.465,0.535,0.000,14.520,2.918,1360.210,386.443,1.000
4,claude-3-5-sonnet-20241022,few,gpqa,0.518,0.482,0.000,23.637,2.845,5055.460,382.926,1.000
17,claude-3-5-sonnet-20241022,zero,gpqa,0.422,0.578,0.000,19.290,2.803,2161.698,369.970,1.000
18,claude-3-5-sonnet-20241022,baseline,simpleqa,0.528,0.472,0.000,6.341,1.586,453.530,80.980,1.000


In [5]:
cms = {}
for file in glob.glob(f"experiments/{RUN_VERSION}/*.json"):
    match = re.match(r'experiments/.+/(.+)-(baseline|zero|few)-(gpqa|simpleqa|mmlu-pro).json', file)
    model = match.group(1)
    prompt = match.group(2)
    dataset = match.group(3)
    if model not in cms:
        cms[model] = {}
    if prompt not in cms[model]:
        cms[model][prompt] = {}
    df_run = pd.DataFrame.from_records(json.load(open(file, 'r')))
    cms[model][prompt][dataset] = pycm.ConfusionMatrix(df_run["label"].tolist(), df_run["wk_v"].tolist(), digit=2, classes=[ 't', 'f' ])

data = [
    [ 
        model, 
        prompt, 
        dataset, 
        cms[model][prompt][dataset].F1_Macro, 
        cms[model][prompt][dataset].ACC_Macro, 
        cms[model][prompt][dataset].FPR['t'], 
        cms[model][prompt][dataset].FNR['t'], 
        cms[model][prompt][dataset].F1['t'], 
        cms[model][prompt][dataset].F1['f']
    ] 
    for model in cms 
    for prompt in cms[model] 
    for dataset in cms[model][prompt]
]

column_names = ["Judge Model", "Prompt", "Dataset", "Macro-F1", "Acc.", "FPR", "FNR", "F1 (+)", "F1 (-)"]
df2 = pd.DataFrame(data, columns=column_names)
df2 = df2.round(3)
df2 = df2.sort_values(["Judge Model", "Dataset", "Prompt"])
df2

,Judge Model,Prompt,Dataset,Macro-F1,Acc.,FPR,FNR,F1 (+),F1 (-)
34,claude-3-5-haiku-20241022,baseline,gpqa,0.533,0.535,0.546,0.378,0.563,0.503
30,claude-3-5-haiku-20241022,few,gpqa,0.577,0.579,0.512,0.323,0.607,0.546
32,claude-3-5-haiku-20241022,zero,gpqa,0.556,0.565,0.594,0.264,0.620,0.491
35,claude-3-5-haiku-20241022,baseline,simpleqa,0.595,0.600,0.281,0.517,0.548,0.641
31,claude-3-5-haiku-20241022,few,simpleqa,0.523,0.578,0.080,0.761,0.362,0.684
33,claude-3-5-haiku-20241022,zero,simpleqa,0.550,0.580,0.156,0.682,0.432,0.667
23,claude-3-5-sonnet-20241022,baseline,gpqa,0.667,0.668,0.372,0.290,0.673,0.662
18,claude-3-5-sonnet-20241022,few,gpqa,0.660,0.660,0.329,0.352,0.648,0.671
20,claude-3-5-sonnet-20241022,zero,gpqa,0.664,0.665,0.415,0.249,0.684,0.644
22,claude-3-5-sonnet-20241022,baseline,simpleqa,0.705,0.705,0.266,0.323,0.697,0.712


In [6]:
df = df2.merge(df1, on=['Judge Model', 'Prompt', 'Dataset'], how='inner')
df

,Judge Model,Prompt,Dataset,Macro-F1,Acc.,FPR,FNR,F1 (+),F1 (-),f,t,e,Time (mean),Time (stdev),Tokens (mean),Tokens (stdev),Coverage
0,claude-3-5-haiku-20241022,baseline,gpqa,0.533,0.535,0.546,0.378,0.563,0.503,0.418,0.582,0.000,14.249,2.235,1346.538,377.096,1.000
1,claude-3-5-haiku-20241022,few,gpqa,0.577,0.579,0.512,0.323,0.607,0.546,0.408,0.590,0.002,19.476,1.553,4829.188,327.028,0.998
2,claude-3-5-haiku-20241022,zero,gpqa,0.556,0.565,0.594,0.264,0.620,0.491,0.338,0.662,0.000,19.696,2.435,2070.332,328.058,1.000
3,claude-3-5-haiku-20241022,baseline,simpleqa,0.595,0.600,0.281,0.517,0.548,0.641,0.618,0.382,0.000,6.687,1.598,489.725,91.756,1.000
4,claude-3-5-haiku-20241022,few,simpleqa,0.523,0.578,0.080,0.761,0.362,0.684,0.840,0.160,0.000,17.023,1.548,4332.010,63.458,1.000
5,claude-3-5-haiku-20241022,zero,simpleqa,0.550,0.580,0.156,0.682,0.432,0.667,0.762,0.238,0.000,16.287,3.868,1505.905,93.596,1.000
6,claude-3-5-sonnet-20241022,baseline,gpqa,0.667,0.668,0.372,0.290,0.673,0.662,0.465,0.535,0.000,14.520,2.918,1360.210,386.443,1.000
7,claude-3-5-sonnet-20241022,few,gpqa,0.660,0.660,0.329,0.352,0.648,0.671,0.518,0.482,0.000,23.637,2.845,5055.460,382.926,1.000
8,claude-3-5-sonnet-20241022,zero,gpqa,0.664,0.665,0.415,0.249,0.684,0.644,0.422,0.578,0.000,19.290,2.803,2161.698,369.970,1.000
9,claude-3-5-sonnet-20241022,baseline,simpleqa,0.705,0.705,0.266,0.323,0.697,0.712,0.528,0.472,0.000,6.341,1.586,453.530,80.980,1.000


In [7]:
df_evaluation = df[["Dataset", "Judge Model", "Prompt", "Coverage", "Macro-F1", "Acc.", "FPR", "FNR", "F1 (+)", "F1 (-)"]].copy()
df_evaluation = df_evaluation.sort_values(["Dataset", "Judge Model", "Prompt"])
df_evaluation

,Dataset,Judge Model,Prompt,Coverage,Macro-F1,Acc.,FPR,FNR,F1 (+),F1 (-)
0,gpqa,claude-3-5-haiku-20241022,baseline,1.000,0.533,0.535,0.546,0.378,0.563,0.503
1,gpqa,claude-3-5-haiku-20241022,few,0.998,0.577,0.579,0.512,0.323,0.607,0.546
2,gpqa,claude-3-5-haiku-20241022,zero,1.000,0.556,0.565,0.594,0.264,0.620,0.491
6,gpqa,claude-3-5-sonnet-20241022,baseline,1.000,0.667,0.668,0.372,0.290,0.673,0.662
7,gpqa,claude-3-5-sonnet-20241022,few,1.000,0.660,0.660,0.329,0.352,0.648,0.671
8,gpqa,claude-3-5-sonnet-20241022,zero,1.000,0.664,0.665,0.415,0.249,0.684,0.644
12,gpqa,llama-4-maverick,baseline,1.000,0.722,0.722,0.275,0.280,0.715,0.730
13,gpqa,llama-4-maverick,few,0.990,0.717,0.717,0.288,0.277,0.711,0.723
14,gpqa,llama-4-maverick,zero,0.998,0.694,0.694,0.296,0.316,0.684,0.704
18,gpqa,llama-4-scout,baseline,1.000,0.636,0.650,0.184,0.528,0.565,0.707


In [8]:
df_evaluation.mean(numeric_only=True)

Coverage    0.999111
Macro-F1    0.598917
Acc.        0.613778
FPR         0.327806
FNR         0.447444
F1 (+)      0.564944
F1 (-)      0.632694
dtype: float64

In [9]:
df_evaluation[df_evaluation["Dataset"] == "gpqa"].mean(numeric_only=True)

Coverage    0.999111
Macro-F1    0.591611
Acc.        0.611167
FPR         0.298444
FNR         0.485500
F1 (+)      0.536444
F1 (-)      0.646500
dtype: float64

In [10]:
df_evaluation[df_evaluation["Dataset"] == "simpleqa"].mean(numeric_only=True)

Coverage    0.999111
Macro-F1    0.606222
Acc.        0.616389
FPR         0.357167
FNR         0.409389
F1 (+)      0.593444
F1 (-)      0.618889
dtype: float64

In [ ]:
# evaluation_table_latex = df_evaluation.to_latex(
#     index=False,          # Set to False to hide index
#     float_format=lambda x: f'{x:0.3g}',
#     column_format=None,  # e.g., 'lrc' for left, right, center alignment
#     longtable=False,     # Set to True for tables that span multiple pages
#     caption="Evaluation of different judge models and evaluation prompts on three datasets (N=100).",        # Add a caption to your table
#     label="tab:expeval",          # Add a label for cross-referencing
#     position='htbp'        # e.g., 'h', 't', 'b', 'p' for table positioning
# )
# print(evaluation_table_latex, file=open("paper/evaluation_table.tex", "w+"))

In [11]:
df_cost = df[["Dataset", "Judge Model", "Prompt", 'Time (mean)', 'Time (stdev)', 'Tokens (mean)', 'Tokens (stdev)']].copy()
df_cost["Mean Time"] = df_cost["Time (mean)"].combine(df_cost["Time (stdev)"], lambda mean, sd: f'{mean:6.2f} ({sd:.2f})')
df_cost["Mean Tokens Used"] = df_cost["Tokens (mean)"].combine(df_cost["Tokens (stdev)"], lambda mean, sd: f'{mean:6.2f} ({sd:.2f})')
df_cost = df_cost[["Dataset", "Judge Model", "Prompt", 'Mean Time', 'Mean Tokens Used']]
df_cost = df_cost.sort_values(["Dataset", "Judge Model", "Prompt"])
df_cost

,Dataset,Judge Model,Prompt,Mean Time,Mean Tokens Used
0,gpqa,claude-3-5-haiku-20241022,baseline,14.25 (2.23),1346.54 (377.10)
1,gpqa,claude-3-5-haiku-20241022,few,19.48 (1.55),4829.19 (327.03)
2,gpqa,claude-3-5-haiku-20241022,zero,19.70 (2.44),2070.33 (328.06)
6,gpqa,claude-3-5-sonnet-20241022,baseline,14.52 (2.92),1360.21 (386.44)
7,gpqa,claude-3-5-sonnet-20241022,few,23.64 (2.85),5055.46 (382.93)
8,gpqa,claude-3-5-sonnet-20241022,zero,19.29 (2.80),2161.70 (369.97)
12,gpqa,llama-4-maverick,baseline,26.83 (21.83),2919.78 (825.91)
13,gpqa,llama-4-maverick,few,24.49 (11.06),5756.76 (602.70)
14,gpqa,llama-4-maverick,zero,25.20 (10.44),3462.93 (665.47)
18,gpqa,llama-4-scout,baseline,30.07 (19.93),2467.57 (1096.58)


In [12]:
df_cost_numeric = df[["Dataset", "Judge Model", "Prompt", 'Time (mean)', 'Time (stdev)', 'Tokens (mean)', 'Tokens (stdev)']].copy()

In [13]:
df_cost_numeric.mean(numeric_only=True)

Time (mean)         13.032333
Time (stdev)         6.482056
Tokens (mean)     2399.414583
Tokens (stdev)     389.386556
dtype: float64

In [14]:
df_cost_numeric[(df_cost_numeric["Judge Model"] != "llama-4-scout") & (df_cost_numeric["Judge Model"] != "llama-4-maverick")].mean(numeric_only=True)

Time (mean)         10.644042
Time (stdev)         3.778083
Tokens (mean)     2153.314125
Tokens (stdev)     270.679083
dtype: float64

In [ ]:
# cost_table_latex = df_cost.to_latex(
#     index=False,          # Set to False to hide index
#     float_format=lambda x: f'{x:6.6g}',
#     column_format=None,  # e.g., 'lrc' for left, right, center alignment
#     longtable=False,     # Set to True for tables that span multiple pages
#     caption="Execution time and tokens used by different judge models and evaluation prompts on three datasets (N=100).",        # Add a caption to your table
#     label="tab:expcosts",          # Add a label for cross-referencing
#     position='htbp'        # e.g., 'h', 't', 'b', 'p' for table positioning
# )
# print(cost_table_latex, file=open("paper/cost_table.tex", "w+"))

In [15]:
warnings.filterwarnings('ignore')

def macro_f1_score(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """
    Compute macro F1 score.
    
    Args:
        y_true: Ground truth binary labels
        y_pred: Predicted binary labels
        
    Returns:
        Macro F1 score
    """
    return f1_score(y_true, y_pred, average='macro')

def subsample_standard_error(y_true: np.ndarray, 
                           y_pred: np.ndarray,
                           subsample_size: int = None,
                           n_subsamples: int = 1000,
                           random_state: int = None) -> Tuple[float, float, List[float]]:
    """
    Estimate standard error of macro F1 using subsampling method from Politis & Romano (1994).
    
    Args:
        y_true: Ground truth binary labels (n,)
        y_pred: Predicted binary labels (n,)
        subsample_size: Size of each subsample (b). If None, uses b = n^(2/3)
        n_subsamples: Number of subsamples to draw
        random_state: Random seed for reproducibility
        
    Returns:
        Tuple of (original_f1, estimated_std_error, subsample_f1_scores)
    """
    if random_state is not None:
        np.random.seed(random_state)
    
    n = len(y_true)
    
    # Choose subsample size according to theory: b → ∞ and b/n → 0
    # Common choice is b = n^(2/3) for optimal rate
    if subsample_size is None:
        subsample_size = int(np.power(n, 2/3))
    
    # Ensure subsample size is valid
    subsample_size = min(subsample_size, n-1)
    subsample_size = max(subsample_size, 2)  # Need at least 2 samples for F1
    
    # print(f"Sample size n = {n}")
    # print(f"Subsample size b = {subsample_size}")
    # print(f"Ratio b/n = {subsample_size/n:.3f}")
    # print(f"Number of subsamples = {n_subsamples}")
    
    # Compute original statistic
    original_f1 = macro_f1_score(y_true, y_pred)
    
    # Generate subsamples and compute F1 scores
    subsample_f1_scores = []
    
    for i in range(n_subsamples):
        # Sample without replacement
        indices = np.random.choice(n, size=subsample_size, replace=False)
        y_true_sub = y_true[indices]
        y_pred_sub = y_pred[indices]
        
        # Compute F1 on subsample
        try:
            f1_sub = macro_f1_score(y_true_sub, y_pred_sub)
            subsample_f1_scores.append(f1_sub)
        except:
            # Skip if F1 cannot be computed (e.g., if one class is missing)
            continue
    
    subsample_f1_scores = np.array(subsample_f1_scores)
    
    # According to the paper, we need to properly normalize
    # The standard error is estimated from the variance of the subsample statistics
    # For the empirical variance, we use the sample variance of subsample F1 scores
    
    # Estimate standard error
    # This approximates the standard error of the original F1 score
    estimated_std_error = np.sqrt(subsample_size / n) * np.std(subsample_f1_scores, ddof=1)
    
    # print(f"\nResults:")
    # print(f"Original macro F1: {original_f1:.4f}")
    # print(f"Mean subsample F1: {np.mean(subsample_f1_scores):.4f}")
    # print(f"Std of subsample F1: {np.std(subsample_f1_scores, ddof=1):.4f}")
    # print(f"Estimated standard error: {estimated_std_error:.4f}")
    # print(f"Number of valid subsamples: {len(subsample_f1_scores)}")
    
    return original_f1, estimated_std_error, subsample_f1_scores.tolist()

def confidence_interval(f1_score: float, 
                       std_error: float, 
                       confidence_level: float = 0.95) -> Tuple[float, float]:
    """
    Construct confidence interval for F1 score using normal approximation.
    
    Args:
        f1_score: Original F1 score
        std_error: Estimated standard error
        confidence_level: Confidence level (default 0.95 for 95% CI)
        
    Returns:
        Tuple of (lower_bound, upper_bound)
    """
    from scipy.stats import norm
    
    alpha = 1 - confidence_level
    z_score = norm.ppf(1 - alpha/2)
    
    margin_of_error = z_score * std_error
    lower_bound = max(0, f1_score - margin_of_error)  # F1 is bounded by 0
    upper_bound = min(1, f1_score + margin_of_error)  # F1 is bounded by 1
    
    return lower_bound, upper_bound

def analyze_classification_results(df, 
                                 true_label_col='ground_truth', 
                                 pred_label_col='predictions',
                                 subsample_size=None,
                                 n_subsamples=1000,
                                 random_state=42):
    """
    Apply subsampling standard error estimation to your classification DataFrame.
    
    Args:
        df: pandas DataFrame with your classification results
        true_label_col: name of column containing ground truth labels
        pred_label_col: name of column containing predictions
        subsample_size: size of subsamples (if None, uses n^(2/3))
        n_subsamples: number of subsamples to draw
        random_state: random seed
        
    Returns:
        Dictionary with results
    """
    
    # Extract arrays from DataFrame
    y_true = df[true_label_col].values
    y_pred = df[pred_label_col].values
    
    # Ensure binary values
    assert set(np.unique(y_true)).issubset({0, 1}), "Ground truth must be binary (0/1)"
    assert set(np.unique(y_pred)).issubset({0, 1}), "Predictions must be binary (0/1)"
    
    # print(f"Analyzing {len(df)} classification results...")
    # print(f"Class distribution in ground truth: {np.bincount(y_true)}")
    # print(f"Class distribution in predictions: {np.bincount(y_pred)}")
    
    # Apply subsampling method
    f1_score, std_error, subsample_scores = subsample_standard_error(
        y_true, y_pred,
        subsample_size=subsample_size,
        n_subsamples=n_subsamples,
        random_state=random_state
    )
    
    # Get confidence interval
    ci_lower, ci_upper = confidence_interval(f1_score, std_error, 0.95)
    
    results = {
        'macro_f1': f1_score,
        'standard_error': std_error,
        'confidence_interval_95': (ci_lower, ci_upper),
        'subsample_scores': subsample_scores,
        'n_valid_subsamples': len(subsample_scores)
    }
    
    return results

# Example usage with your DataFrame:
def example_usage():
    """
    Example of how to use with your DataFrame
    """
    # If your DataFrame looks like this:
    #   ground_truth  predictions
    # 0            1            1
    # 1            0            1
    # 2            1            0
    # ... etc for 400 rows

    # Load your data (replace with your actual loading method)
    # df = pd.read_csv('your_data.csv')
    
    # Create sample data for demonstration
    np.random.seed(42)
    n = 400
    sample_data = {
        'ground_truth': np.random.choice([0, 1], size=n, p=[0.3, 0.7]),
        'predictions': np.random.choice([0, 1], size=n, p=[0.4, 0.6])
    }
    df = pd.DataFrame(sample_data)
    
    # Run the analysis
    results = analyze_classification_results(
        df, 
        true_label_col='ground_truth',    # adjust column name as needed
        pred_label_col='predictions',     # adjust column name as needed
        n_subsamples=1000,               # as you requested
        random_state=42                  # for reproducibility
    )
    
    # Print results
    print(f"\n=== FINAL RESULTS ===")
    print(f"Macro F1 Score: {results['macro_f1']:.4f}")
    print(f"Standard Error: {results['standard_error']:.4f}")
    print(f"95% CI: [{results['confidence_interval_95'][0]:.4f}, {results['confidence_interval_95'][1]:.4f}]")
    
    return results

# Alternative: if you want to experiment with different subsample sizes
def compare_subsample_sizes(df, true_label_col, pred_label_col, sizes=None):
    """
    Compare results using different subsample sizes to see sensitivity.
    """
    if sizes is None:
        n = len(df)
        sizes = [
            int(n**0.5),     # n^(1/2) 
            int(n**(2/3)),   # n^(2/3) - theoretical optimum
            int(n**0.8),     # n^(4/5)
            n//4,            # n/4
            n//3             # n/3
        ]
    
    results = {}
    y_true = df[true_label_col].values
    y_pred = df[pred_label_col].values
    
    for size in sizes:
        if size >= len(df) or size < 10:
            continue
            
        print(f"\n--- Subsample size: {size} (ratio: {size/len(df):.3f}) ---")
        f1, se, _ = subsample_standard_error(
            y_true, y_pred, 
            subsample_size=size, 
            n_subsamples=1000,
            random_state=42
        )
        results[size] = {'f1': f1, 'std_error': se}
    
    return results

if __name__ == "__main__":
    # Run the example
    example_results = example_usage()


=== FINAL RESULTS ===
Macro F1 Score: 0.4483
Standard Error: 0.0229
95% CI: [0.4034, 0.4932]


In [16]:
cms = {}
for file in glob.glob(f"experiments/{RUN_VERSION}/*.json"):
    match = re.match(r'experiments/.+/(.+)-(baseline|zero|few)-(gpqa|simpleqa|mmlu-pro).json', file)
    model = match.group(1)
    prompt = match.group(2)
    dataset = match.group(3)
    if model not in cms:
        cms[model] = {}
    if prompt not in cms[model]:
        cms[model][prompt] = {}
    df_run = pd.DataFrame.from_records(json.load(open(file, 'r')))
    df_run = df_run[(df_run["wk_v"] == 't') | (df_run["wk_v"] == 'f')].copy()
    df_run["ground_truth"] = (df_run["label"] == 't').astype(int)
    df_run["predictions"] = (df_run["wk_v"] == 't').astype(int)
    cms[model][prompt][dataset] = analyze_classification_results(df_run, true_label_col='ground_truth', pred_label_col='predictions', n_subsamples=1000, random_state=9331)

data = [
    [ 
        model, 
        prompt, 
        dataset, 
        cms[model][prompt][dataset]['macro_f1'], 
        cms[model][prompt][dataset]['standard_error'],
    ] 
    for model in cms 
    for prompt in cms[model] 
    for dataset in cms[model][prompt]
]

column_names = ["Judge Model", "Prompt", "Dataset", "Macro-F1", "SE"]
df_subsamples = pd.DataFrame(data, columns=column_names)
df_subsamples = df_subsamples.round(3)
df_subsamples = df_subsamples.sort_values(["Judge Model", "Prompt", "Dataset"]).copy()
# df_subsamples = df_subsamples[["Judge Model", "Prompt", "Dataset", "Macro-F1", "SE"]]
df_subsamples

,Judge Model,Prompt,Dataset,Macro-F1,SE
34,claude-3-5-haiku-20241022,baseline,gpqa,0.533,0.023
35,claude-3-5-haiku-20241022,baseline,simpleqa,0.595,0.023
30,claude-3-5-haiku-20241022,few,gpqa,0.577,0.023
31,claude-3-5-haiku-20241022,few,simpleqa,0.523,0.024
32,claude-3-5-haiku-20241022,zero,gpqa,0.556,0.024
33,claude-3-5-haiku-20241022,zero,simpleqa,0.550,0.025
23,claude-3-5-sonnet-20241022,baseline,gpqa,0.667,0.021
22,claude-3-5-sonnet-20241022,baseline,simpleqa,0.705,0.022
18,claude-3-5-sonnet-20241022,few,gpqa,0.660,0.022
19,claude-3-5-sonnet-20241022,few,simpleqa,0.661,0.023


In [17]:
# Assuming your dataframe is called 'df'
# If reading from a file, uncomment and modify the appropriate line below:
# df = pd.read_csv('your_file.csv')
# df = pd.read_excel('your_file.xlsx')

def transform_dataframe(df):
    """
    Transform the dataframe from long format to wide format.
    
    Original format: Judge Model, Prompt, Dataset, Macro-F1, SE
    New format: Judge Model, Prompt, GQPA_F1, GQPA_SE, SimpleQA_F1, SimpleQA_SE
    """
    
    # Create the pivot table
    # We'll pivot on the Dataset column to create separate columns for each dataset
    transformed_df = df.pivot_table(
        index=['Judge Model', 'Prompt'],  # These will be our row identifiers
        columns='Dataset',                # This will become our column headers
        values=['Macro-F1', 'SE'],       # These are the values we want to pivot
        aggfunc='first'                  # In case of duplicates, take the first value
    ).reset_index()
    
    # Flatten the multi-level column headers
    # This will create column names like ('Macro-F1', 'gqpa'), ('Macro-F1', 'simpleqa'), etc.
    transformed_df.columns = [f'{col[1]}_{col[0]}' if col[1] != '' else col[0] 
                             for col in transformed_df.columns]
    
    # Rename columns to match your desired format
    column_mapping = {
        'Judge Model': 'Judge Model',
        'Prompt': 'Prompt',
        'gpqa_Macro-F1': 'GPQA_F1',
        'gpqa_SE': 'GPQA_F1_SE',
        'simpleqa_Macro-F1': 'SimpleQA_F1',
        'simpleqa_SE': 'SimpleQA_F1_SE'
    }
    
    transformed_df = transformed_df.rename(columns=column_mapping)
    
    # Reorder columns to match your specification
    final_columns = ['Judge Model', 'Prompt', 'GPQA_F1', 'GPQA_F1_SE', 'SimpleQA_F1', 'SimpleQA_F1_SE']
    transformed_df = transformed_df[final_columns]
    
    return transformed_df

# Apply the transformation
# transformed_df = transform_dataframe(df)

# Display the result
# print(transformed_df)

# Save to file if needed
# transformed_df.to_csv('transformed_data.csv', index=False)
# transformed_df.to_excel('transformed_data.xlsx', index=False)

# Example of what the output will look like:
"""
Expected output format:
        Judge Model  Prompt  GQPA_F1  GQPA_SE  SimpleQA_F1  SimpleQA_SE
0   claude-3-5-haiku-20241022  baseline    0.578    0.027        0.667      0.033
1   claude-3-5-haiku-20241022       few    0.604    0.034        0.653      0.033
2   claude-3-5-haiku-20241022      zero    0.648    0.034        0.673      0.036
3  claude-3-5-sonnet-20241022  baseline    0.712    0.023        0.810      0.025
...
"""

'\nExpected output format:\n        Judge Model  Prompt  GQPA_F1  GQPA_SE  SimpleQA_F1  SimpleQA_SE\n0   claude-3-5-haiku-20241022  baseline    0.578    0.027        0.667      0.033\n1   claude-3-5-haiku-20241022       few    0.604    0.034        0.653      0.033\n2   claude-3-5-haiku-20241022      zero    0.648    0.034        0.673      0.036\n3  claude-3-5-sonnet-20241022  baseline    0.712    0.023        0.810      0.025\n...\n'

In [18]:
df_transformed_f1 = transform_dataframe(df_subsamples)

In [19]:
coverage = {}
for file in glob.glob(f"experiments/{RUN_VERSION}/*.json"):
    match = re.match(r'experiments/.+/(.+)-(baseline|zero|few)-(gpqa|simpleqa|mmlu-pro).json', file)
    model = match.group(1)
    prompt = match.group(2)
    dataset = match.group(3)
    if model not in coverage:
        coverage[model] = {}
    if prompt not in coverage[model]:
        coverage[model][prompt] = {}
    if dataset not in coverage[model][prompt]:
        coverage[model][prompt][dataset] = {}
    df_run = pd.DataFrame.from_records(json.load(open(file, 'r')))
    df_run["coverage"] = (df_run["wk_v"] != 'e').astype(int)
    coverage[model][prompt][dataset] = subsample_statistic_standard_error(df_run["coverage"].to_numpy(), np.mean, n_subsamples=1000, random_state=9331)

data = [ 
    [
        model, 
        prompt, 
        dataset, 
        coverage[model][prompt][dataset][0], 
        coverage[model][prompt][dataset][1]
    ] 
    for model in coverage 
    for prompt in coverage[model] 
    for dataset in coverage[model][prompt]
]

column_names = ["Judge Model", "Prompt", "Dataset", "Coverage", "SE"]
df_coverage_2 = pd.DataFrame(data, columns=column_names)
df_coverage_2 = df_coverage_2.round(3)
df_coverage_2 = df_coverage_2.sort_values(["Judge Model", "Prompt", "Dataset"]).copy()
df_coverage_2

,Judge Model,Prompt,Dataset,Coverage,SE
34,claude-3-5-haiku-20241022,baseline,gpqa,1.000,0.000
35,claude-3-5-haiku-20241022,baseline,simpleqa,1.000,0.000
30,claude-3-5-haiku-20241022,few,gpqa,0.998,0.002
31,claude-3-5-haiku-20241022,few,simpleqa,1.000,0.000
32,claude-3-5-haiku-20241022,zero,gpqa,1.000,0.000
33,claude-3-5-haiku-20241022,zero,simpleqa,1.000,0.000
23,claude-3-5-sonnet-20241022,baseline,gpqa,1.000,0.000
22,claude-3-5-sonnet-20241022,baseline,simpleqa,1.000,0.000
18,claude-3-5-sonnet-20241022,few,gpqa,1.000,0.000
19,claude-3-5-sonnet-20241022,few,simpleqa,1.000,0.000


In [20]:
transformed_df_coverage = df_coverage_2.pivot_table(
        index=['Judge Model', 'Prompt'],  # These will be our row identifiers
        columns='Dataset',                # This will become our column headers
        values=['Coverage', 'SE'],       # These are the values we want to pivot
        aggfunc='first'                  # In case of duplicates, take the first value
    ).reset_index()

# Flatten the multi-level column headers
# This will create column names like ('Macro-F1', 'gqpa'), ('Macro-F1', 'simpleqa'), etc.
transformed_df_coverage.columns = [f'{col[1]}_{col[0]}' if col[1] != '' else col[0] 
                            for col in transformed_df_coverage.columns]
    
# Rename columns to match your desired format
column_mapping = {
    'Judge Model': 'Judge Model',
    'Prompt': 'Prompt',
    'gpqa_Coverage': 'GPQA_Coverage',
    'gpqa_SE': 'GPQA_Coverage_SE',
    'simpleqa_Coverage': 'SimpleQA_Coverage',
    'simpleqa_SE': 'SimpleQA_Coverage_SE'
}
    
transformed_df_coverage = transformed_df_coverage.rename(columns=column_mapping)
transformed_df_coverage


,Judge Model,Prompt,GPQA_Coverage,SimpleQA_Coverage,GPQA_Coverage_SE,SimpleQA_Coverage_SE
0,claude-3-5-haiku-20241022,baseline,1.000,1.000,0.000,0.000
1,claude-3-5-haiku-20241022,few,0.998,1.000,0.002,0.000
2,claude-3-5-haiku-20241022,zero,1.000,1.000,0.000,0.000
3,claude-3-5-sonnet-20241022,baseline,1.000,1.000,0.000,0.000
4,claude-3-5-sonnet-20241022,few,1.000,1.000,0.000,0.000
5,claude-3-5-sonnet-20241022,zero,1.000,1.000,0.000,0.000
6,llama-4-maverick,baseline,1.000,1.000,0.000,0.000
7,llama-4-maverick,few,0.990,0.992,0.005,0.004
8,llama-4-maverick,zero,0.998,0.992,0.002,0.004
9,llama-4-scout,baseline,1.000,1.000,0.000,0.000


In [21]:
df_perf_merged = pd.merge(df_transformed_f1, transformed_df_coverage)

In [22]:
df_perf_merged

,Judge Model,Prompt,GPQA_F1,GPQA_F1_SE,SimpleQA_F1,SimpleQA_F1_SE,GPQA_Coverage,SimpleQA_Coverage,GPQA_Coverage_SE,SimpleQA_Coverage_SE
0,claude-3-5-haiku-20241022,baseline,0.533,0.023,0.595,0.023,1.000,1.000,0.000,0.000
1,claude-3-5-haiku-20241022,few,0.577,0.023,0.523,0.024,0.998,1.000,0.002,0.000
2,claude-3-5-haiku-20241022,zero,0.556,0.024,0.550,0.025,1.000,1.000,0.000,0.000
3,claude-3-5-sonnet-20241022,baseline,0.667,0.021,0.705,0.022,1.000,1.000,0.000,0.000
4,claude-3-5-sonnet-20241022,few,0.660,0.022,0.661,0.023,1.000,1.000,0.000,0.000
5,claude-3-5-sonnet-20241022,zero,0.664,0.022,0.682,0.022,1.000,1.000,0.000,0.000
6,llama-4-maverick,baseline,0.722,0.021,0.643,0.023,1.000,1.000,0.000,0.000
7,llama-4-maverick,few,0.717,0.021,0.648,0.023,0.990,0.992,0.005,0.004
8,llama-4-maverick,zero,0.694,0.020,0.663,0.023,0.998,0.992,0.002,0.004
9,llama-4-scout,baseline,0.636,0.023,0.578,0.023,1.000,1.000,0.000,0.000


In [23]:
df_perf_merged["GPQA_Cov_disp"] = df_perf_merged.apply(lambda row: f'{row["GPQA_Coverage"]} ({row["GPQA_Coverage_SE"]})', axis=1)
df_perf_merged["GPQA_F1_disp"] = df_perf_merged.apply(lambda row: f'{row["GPQA_F1"]} ({row["GPQA_F1_SE"]})', axis=1)
df_perf_merged["SimpleQA_Cov_disp"] = df_perf_merged.apply(lambda row: f'{row["SimpleQA_Coverage"]} ({row["SimpleQA_Coverage_SE"]})', axis=1)
df_perf_merged["SimpleQA_F1_disp"] = df_perf_merged.apply(lambda row: f'{row["SimpleQA_F1"]} ({row["SimpleQA_F1_SE"]})', axis=1)

In [24]:
df_perf_merged = df_perf_merged[["Judge Model","Prompt","GPQA_F1_disp", "GPQA_Cov_disp", "SimpleQA_F1_disp", "SimpleQA_Cov_disp"]].copy()

In [25]:
df_perf_merged

,Judge Model,Prompt,GPQA_F1_disp,GPQA_Cov_disp,SimpleQA_F1_disp,SimpleQA_Cov_disp
0,claude-3-5-haiku-20241022,baseline,0.533 (0.023),1.0 (0.0),0.595 (0.023),1.0 (0.0)
1,claude-3-5-haiku-20241022,few,0.577 (0.023),0.998 (0.002),0.523 (0.024),1.0 (0.0)
2,claude-3-5-haiku-20241022,zero,0.556 (0.024),1.0 (0.0),0.55 (0.025),1.0 (0.0)
3,claude-3-5-sonnet-20241022,baseline,0.667 (0.021),1.0 (0.0),0.705 (0.022),1.0 (0.0)
4,claude-3-5-sonnet-20241022,few,0.66 (0.022),1.0 (0.0),0.661 (0.023),1.0 (0.0)
5,claude-3-5-sonnet-20241022,zero,0.664 (0.022),1.0 (0.0),0.682 (0.022),1.0 (0.0)
6,llama-4-maverick,baseline,0.722 (0.021),1.0 (0.0),0.643 (0.023),1.0 (0.0)
7,llama-4-maverick,few,0.717 (0.021),0.99 (0.005),0.648 (0.023),0.992 (0.004)
8,llama-4-maverick,zero,0.694 (0.02),0.998 (0.002),0.663 (0.023),0.992 (0.004)
9,llama-4-scout,baseline,0.636 (0.023),1.0 (0.0),0.578 (0.023),1.0 (0.0)


In [26]:
transformed_df_cost = df_cost.pivot_table(
        index=['Judge Model', 'Prompt'],  # These will be our row identifiers
        columns='Dataset',                # This will become our column headers
        values=['Mean Time', 'Mean Tokens Used'],       # These are the values we want to pivot
        aggfunc='first'                  # In case of duplicates, take the first value
    ).reset_index()

# Flatten the multi-level column headers
# This will create column names like ('Macro-F1', 'gqpa'), ('Macro-F1', 'simpleqa'), etc.
transformed_df_cost.columns = [f'{col[1]}_{col[0]}' if col[1] != '' else col[0] 
                            for col in transformed_df_cost.columns]
    
# Rename columns to match your desired format
column_mapping = {
    'Judge Model': 'Judge Model',
    'Prompt': 'Prompt',
    'gpqa_Mean Time': 'GPQA_Mean_Time',
    'gpqa_Mean Tokens Used': 'GPQA_Mean_Tokens',
    'simpleqa_Mean Time': 'SimpleQA_Mean_Time',
    'simpleqa_Mean Tokens Used': 'SimpleQA_Mean_Tokens',
}
    
transformed_df_cost = transformed_df_cost.rename(columns=column_mapping)
transformed_df_cost


,Judge Model,Prompt,GPQA_Mean_Time,SimpleQA_Mean_Time,GPQA_Mean_Tokens,SimpleQA_Mean_Tokens
0,claude-3-5-haiku-20241022,baseline,14.25 (2.23),6.69 (1.60),1346.54 (377.10),489.73 (91.76)
1,claude-3-5-haiku-20241022,few,19.48 (1.55),17.02 (1.55),4829.19 (327.03),4332.01 (63.46)
2,claude-3-5-haiku-20241022,zero,19.70 (2.44),16.29 (3.87),2070.33 (328.06),1505.90 (93.60)
3,claude-3-5-sonnet-20241022,baseline,14.52 (2.92),6.34 (1.59),1360.21 (386.44),453.53 (80.98)
4,claude-3-5-sonnet-20241022,few,23.64 (2.85),16.82 (1.99),5055.46 (382.93),4331.51 (89.63)
5,claude-3-5-sonnet-20241022,zero,19.29 (2.80),16.28 (5.05),2161.70 (369.97),1499.60 (112.18)
6,llama-4-maverick,baseline,26.83 (21.83),6.71 (4.20),2919.78 (825.91),814.69 (350.94)
7,llama-4-maverick,few,24.49 (11.06),16.97 (10.19),5756.76 (602.70),4524.26 (265.16)
8,llama-4-maverick,zero,25.20 (10.44),14.88 (8.78),3462.93 (665.47),2097.99 (230.58)
9,llama-4-scout,baseline,30.07 (19.93),5.73 (4.96),2467.57 (1096.58),511.43 (286.82)


In [27]:
df_perf_merged = pd.merge(df_perf_merged, transformed_df_cost)

In [28]:
df_perf_merged

,Judge Model,Prompt,GPQA_F1_disp,GPQA_Cov_disp,SimpleQA_F1_disp,SimpleQA_Cov_disp,GPQA_Mean_Time,SimpleQA_Mean_Time,GPQA_Mean_Tokens,SimpleQA_Mean_Tokens
0,claude-3-5-haiku-20241022,baseline,0.533 (0.023),1.0 (0.0),0.595 (0.023),1.0 (0.0),14.25 (2.23),6.69 (1.60),1346.54 (377.10),489.73 (91.76)
1,claude-3-5-haiku-20241022,few,0.577 (0.023),0.998 (0.002),0.523 (0.024),1.0 (0.0),19.48 (1.55),17.02 (1.55),4829.19 (327.03),4332.01 (63.46)
2,claude-3-5-haiku-20241022,zero,0.556 (0.024),1.0 (0.0),0.55 (0.025),1.0 (0.0),19.70 (2.44),16.29 (3.87),2070.33 (328.06),1505.90 (93.60)
3,claude-3-5-sonnet-20241022,baseline,0.667 (0.021),1.0 (0.0),0.705 (0.022),1.0 (0.0),14.52 (2.92),6.34 (1.59),1360.21 (386.44),453.53 (80.98)
4,claude-3-5-sonnet-20241022,few,0.66 (0.022),1.0 (0.0),0.661 (0.023),1.0 (0.0),23.64 (2.85),16.82 (1.99),5055.46 (382.93),4331.51 (89.63)
5,claude-3-5-sonnet-20241022,zero,0.664 (0.022),1.0 (0.0),0.682 (0.022),1.0 (0.0),19.29 (2.80),16.28 (5.05),2161.70 (369.97),1499.60 (112.18)
6,llama-4-maverick,baseline,0.722 (0.021),1.0 (0.0),0.643 (0.023),1.0 (0.0),26.83 (21.83),6.71 (4.20),2919.78 (825.91),814.69 (350.94)
7,llama-4-maverick,few,0.717 (0.021),0.99 (0.005),0.648 (0.023),0.992 (0.004),24.49 (11.06),16.97 (10.19),5756.76 (602.70),4524.26 (265.16)
8,llama-4-maverick,zero,0.694 (0.02),0.998 (0.002),0.663 (0.023),0.992 (0.004),25.20 (10.44),14.88 (8.78),3462.93 (665.47),2097.99 (230.58)
9,llama-4-scout,baseline,0.636 (0.023),1.0 (0.0),0.578 (0.023),1.0 (0.0),30.07 (19.93),5.73 (4.96),2467.57 (1096.58),511.43 (286.82)


In [ ]:
final_df_perf = df_perf_merged[["Judge Model", "Prompt", "GPQA_F1_disp", "GPQA_Cov_disp", "GPQA_Mean_Time", "GPQA_Mean_Tokens", "SimpleQA_F1_disp", "SimpleQA_Cov_disp", "SimpleQA_Mean_Time", "SimpleQA_Mean_Tokens"]].copy()

In [ ]:
# perf_table_latex = final_df_perf.to_latex(
#     index=False,          # Set to False to hide index
#     float_format=lambda x: f'{x:6.6g}',
#     column_format=None,  # e.g., 'lrc' for left, right, center alignment
#     longtable=False,     # Set to True for tables that span multiple pages
#     caption="Performance metrics for zeta using different judge models and evaluation prompts on three datasets (N=100).",        # Add a caption to your table
#     label="tab:finalperf",          # Add a label for cross-referencing
#     position='htbp'        # e.g., 'h', 't', 'b', 'p' for table positioning
# )
# print(perf_table_latex, file=open("paper/perf_table.tex", "w+"))

In [29]:
summary_df = df[["Judge Model", "Dataset", "Macro-F1", "Coverage", "Time (mean)", "Tokens (mean)"]]

In [30]:
summary_flagship_df = summary_df[(summary_df["Judge Model"] == "claude-3-5-sonnet-20241022") | (summary_df["Judge Model"] == "llama-4-maverick") | (summary_df["Judge Model"] == "nf-gpt-4o")].copy()

In [31]:
summary_distilled_df = summary_df[(summary_df["Judge Model"] == "claude-3-5-haiku-20241022") | (summary_df["Judge Model"] == "llama-4-scout") | (summary_df["Judge Model"] == "nf-gpt-4o-mini")].copy()

In [32]:
summary_flagship_gpqa_series = summary_flagship_df[(df["Dataset"] == 'gpqa')].mean(numeric_only=True)
summary_flagship_gpqa_series["Model Type"] = "flagship"
summary_flagship_gpqa_series["Dataset"] = "gpqa"

In [33]:
summary_flagship_simpleqa_series = summary_flagship_df[(df["Dataset"] == 'simpleqa')].mean(numeric_only=True)
summary_flagship_simpleqa_series["Model Type"] = "flagship"
summary_flagship_simpleqa_series["Dataset"] = "simpleqa"

In [34]:
summary_distilled_gpqa_series = summary_distilled_df[(df["Dataset"] == 'gpqa')].mean(numeric_only=True)
summary_distilled_gpqa_series["Model Type"] = "distilled"
summary_distilled_gpqa_series["Dataset"] = "gpqa"

In [35]:
summary_distilled_simpleqa_series = summary_distilled_df[(df["Dataset"] == 'simpleqa')].mean(numeric_only=True)
summary_distilled_simpleqa_series["Model Type"] = "distilled"
summary_distilled_simpleqa_series["Dataset"] = "simpleqa"

In [36]:
summary_table_df = pd.concat([summary_flagship_gpqa_series, summary_distilled_gpqa_series, summary_flagship_simpleqa_series, summary_distilled_simpleqa_series], axis=1)

In [38]:
summary_table_df = summary_table_df.T[["Dataset", "Model Type", "Macro-F1", "Coverage", "Time (mean)", "Tokens (mean)"]].copy()

In [39]:
summary_table_df

,Dataset,Model Type,Macro-F1,Coverage,Time (mean),Tokens (mean)
0,gpqa,flagship,0.630778,0.998667,16.531444,2863.302667
1,gpqa,distilled,0.552444,0.999556,17.793111,2952.718778
2,simpleqa,flagship,0.652,0.998222,10.157556,1973.750778
3,simpleqa,distilled,0.560444,1.0,7.647222,1807.886111


In [ ]:
# summary_table_latex = summary_table_df.to_latex(
#     index=False,          # Set to False to hide index
#     float_format=lambda x: f'{x:0.4g}',
#     column_format=None,  # e.g., 'lrc' for left, right, center alignment
#     longtable=False,     # Set to True for tables that span multiple pages
#     caption="Summary performance metrics for zeta.",        # Add a caption to your table
#     label="tab:summaryperf",          # Add a label for cross-referencing
#     position='htbp'        # e.g., 'h', 't', 'b', 'p' for table positioning
# )
# print(summary_table_latex, file=open("paper/summary_table.tex", "w+"))

In [40]:
cms = {}
df_runs = []
for file in glob.glob(f"experiments/{RUN_VERSION}/*.json"):
    match = re.match(r'experiments/.+/(.+)-(baseline|zero|few)-(gpqa|simpleqa|mmlu-pro).json', file)
    model = match.group(1)
    prompt = match.group(2)
    dataset = match.group(3)
    if model not in cms:
        cms[model] = {}
    if prompt not in cms[model]:
        cms[model][prompt] = {}
    df_run = pd.DataFrame.from_records(json.load(open(file, 'r')))
    df_run = df_run[(df_run["wk_v"] == 't') | (df_run["wk_v"] == 'f')].copy()
    df_run["ground_truth"] = (df_run["label"] == 't').astype(int)
    df_run["predictions"] = (df_run["wk_v"] == 't').astype(int)
    df_runs.append(df_run.copy())

df_all_runs = pd.concat(df_runs, axis=0)

df_all_runs_gpqa = df_all_runs[(df_all_runs["dataset_name"] == "gpqa")].copy()
df_all_flagship_runs_gpqa = df_all_runs[(df_all_runs["model_name"] == "claude-3-5-sonnet-20241022") | (df_all_runs["model_name"] == "llama-4-maverick") | (df_all_runs["model_name"] == "nf-gpt-4o") & (df_all_runs["dataset_name"] == "gpqa")].copy()
df_all_distilled_runs_gpqa = df_all_runs[(df_all_runs["model_name"] == "claude-3-5-haiku-20241022") | (df_all_runs["model_name"] == "llama-4-scout") | (df_all_runs["model_name"] == "nf-gpt-4o-mini") & (df_all_runs["dataset_name"] == "gpqa")].copy()
df_all_runs_simpleqa = df_all_runs[(df_all_runs["dataset_name"] == "simpleqa")].copy()
df_all_flagship_runs_simpleqa = df_all_runs[(df_all_runs["model_name"] == "claude-3-5-sonnet-20241022") | (df_all_runs["model_name"] == "llama-4-maverick") | (df_all_runs["model_name"] == "nf-gpt-4o") & (df_all_runs["dataset_name"] == "simpleqa")].copy()
df_all_distilled_runs_simpleqa = df_all_runs[(df_all_runs["model_name"] == "claude-3-5-haiku-20241022") | (df_all_runs["model_name"] == "llama-4-scout") | (df_all_runs["model_name"] == "nf-gpt-4o-mini") & (df_all_runs["dataset_name"] == "simpleqa")].copy()
   
f1_analysis_all_gpqa = analyze_classification_results(df_all_runs_gpqa, true_label_col='ground_truth', pred_label_col='predictions', n_subsamples=1000, random_state=9331)
f1_analysis_flagship_gpqa = analyze_classification_results(df_all_flagship_runs_gpqa, true_label_col='ground_truth', pred_label_col='predictions', n_subsamples=1000, random_state=9331)
f1_analysis_distilled_gpqa = analyze_classification_results(df_all_distilled_runs_gpqa, true_label_col='ground_truth', pred_label_col='predictions', n_subsamples=1000, random_state=9331)
f1_analysis_all_simpleqa = analyze_classification_results(df_all_runs_simpleqa, true_label_col='ground_truth', pred_label_col='predictions', n_subsamples=1000, random_state=9331)
f1_analysis_flagship_simpleqa = analyze_classification_results(df_all_flagship_runs_simpleqa, true_label_col='ground_truth', pred_label_col='predictions', n_subsamples=1000, random_state=9331)
f1_analysis_distilled_simpleqa = analyze_classification_results(df_all_distilled_runs_simpleqa, true_label_col='ground_truth', pred_label_col='predictions', n_subsamples=1000, random_state=9331)

print(f'gpqa all:           {f1_analysis_all_gpqa["macro_f1"]:0.3f} ({f1_analysis_all_gpqa["standard_error"]:0.3f})')
print(f'gpqa flagship:      {f1_analysis_flagship_gpqa["macro_f1"]:0.3f} ({f1_analysis_flagship_gpqa["standard_error"]:0.3f})')
print(f'gpqa distilled:     {f1_analysis_distilled_gpqa["macro_f1"]:0.3f} ({f1_analysis_distilled_gpqa["standard_error"]:0.3f})')
print(f'simpleqa all:       {f1_analysis_all_simpleqa["macro_f1"]:0.3f} ({f1_analysis_all_simpleqa["standard_error"]:0.3f})')
print(f'simpleqa flagship:  {f1_analysis_flagship_simpleqa["macro_f1"]:0.3f} ({f1_analysis_flagship_simpleqa["standard_error"]:0.3f})')
print(f'simpleqa distilled: {f1_analysis_distilled_simpleqa["macro_f1"]:0.3f} ({f1_analysis_distilled_simpleqa["standard_error"]:0.3f})')

gpqa all:           0.606 (0.005)
gpqa flagship:      0.633 (0.007)
gpqa distilled:     0.559 (0.008)
simpleqa all:       0.616 (0.006)
simpleqa flagship:  0.657 (0.008)
simpleqa distilled: 0.570 (0.008)


In [42]:
cms = {}
df_runs = []
for file in glob.glob(f"experiments/{RUN_VERSION}/*.json"):
    match = re.match(r'experiments/.+/(.+)-(baseline|zero|few)-(gpqa|simpleqa|mmlu-pro).json', file)
    model = match.group(1)
    prompt = match.group(2)
    dataset = match.group(3)
    if model not in cms:
        cms[model] = {}
    if prompt not in cms[model]:
        cms[model][prompt] = {}
    df_run = pd.DataFrame.from_records(json.load(open(file, 'r')))
    df_run["coverage"] = (df_run["wk_v"] != 'e').astype(int)
    df_runs.append(df_run.copy())

df_all_runs = pd.concat(df_runs, axis=0)

df_all_runs_gpqa = df_all_runs[(df_all_runs["dataset_name"] == "gpqa")].copy()
df_all_flagship_runs_gpqa = df_all_runs[(df_all_runs["model_name"] == "claude-3-5-sonnet-20241022") | (df_all_runs["model_name"] == "llama-4-maverick") | (df_all_runs["model_name"] == "nf-gpt-4o") & (df_all_runs["dataset_name"] == "gpqa")].copy()
df_all_distilled_runs_gpqa = df_all_runs[(df_all_runs["model_name"] == "claude-3-5-haiku-20241022") | (df_all_runs["model_name"] == "llama-4-scout") | (df_all_runs["model_name"] == "nf-gpt-4o-mini") & (df_all_runs["dataset_name"] == "gpqa")].copy()
df_all_runs_simpleqa = df_all_runs[(df_all_runs["dataset_name"] == "simpleqa")].copy()
df_all_flagship_runs_simpleqa = df_all_runs[(df_all_runs["model_name"] == "claude-3-5-sonnet-20241022") | (df_all_runs["model_name"] == "llama-4-maverick") | (df_all_runs["model_name"] == "nf-gpt-4o") & (df_all_runs["dataset_name"] == "simpleqa")].copy()
df_all_distilled_runs_simpleqa = df_all_runs[(df_all_runs["model_name"] == "claude-3-5-haiku-20241022") | (df_all_runs["model_name"] == "llama-4-scout") | (df_all_runs["model_name"] == "nf-gpt-4o-mini") & (df_all_runs["dataset_name"] == "simpleqa")].copy()
   
coverage_analysis_all_gpqa = subsample_statistic_standard_error(df_all_runs_gpqa["coverage"].to_numpy(), np.mean, n_subsamples=1000, random_state=9331)
coverage_analysis_flagship_gpqa = subsample_statistic_standard_error(df_all_flagship_runs_gpqa["coverage"].to_numpy(), np.mean, n_subsamples=1000, random_state=9331)
coverage_analysis_distilled_gpqa = subsample_statistic_standard_error(df_all_distilled_runs_gpqa["coverage"].to_numpy(), np.mean, n_subsamples=1000, random_state=9331)
coverage_analysis_all_simpleqa = subsample_statistic_standard_error(df_all_runs_simpleqa["coverage"].to_numpy(), np.mean, n_subsamples=1000, random_state=9331)
coverage_analysis_flagship_simpleqa = subsample_statistic_standard_error(df_all_flagship_runs_simpleqa["coverage"].to_numpy(), np.mean, n_subsamples=1000, random_state=9331)
coverage_analysis_distilled_simpleqa = subsample_statistic_standard_error(df_all_distilled_runs_simpleqa["coverage"].to_numpy(), np.mean, n_subsamples=1000, random_state=9331)

print(f'Coverage gpqa all:           {coverage_analysis_all_gpqa[0]:0.3f} ({coverage_analysis_all_gpqa[1]:0.3f})')
print(f'Coverage gpqa flagship:      {coverage_analysis_flagship_gpqa[0]:0.3f} ({coverage_analysis_flagship_gpqa[1]:0.3f})')
print(f'Coverage gpqa distilled:     {coverage_analysis_distilled_gpqa[0]:0.3f} ({coverage_analysis_distilled_gpqa[1]:0.3f})')
print(f'Coverage simpleqa all:       {coverage_analysis_all_simpleqa[0]:0.3f} ({coverage_analysis_all_simpleqa[1]:0.3f})')
print(f'Coverage simpleqa flagship:  {coverage_analysis_flagship_simpleqa[0]:0.3f} ({coverage_analysis_flagship_simpleqa[1]:0.3f})')
print(f'Coverage simpleqa distilled: {coverage_analysis_distilled_simpleqa[0]:0.3f} ({coverage_analysis_distilled_simpleqa[1]:0.3f})')

Coverage gpqa all:           0.999 (0.000)
Coverage gpqa flagship:      1.000 (0.000)
Coverage gpqa distilled:     1.000 (0.000)
Coverage simpleqa all:       0.999 (0.000)
Coverage simpleqa flagship:  1.000 (0.000)
Coverage simpleqa distilled: 1.000 (0.000)
